In [ ]:
# GUI/app.py
import sys
import os
import threading
import speech_recognition as sr
from flask import Flask, render_template, request, jsonify
from flask_socketio import SocketIO
import nest_asyncio

# Add parent folder to path (works in Jupyter)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from pi_client import send_command_to_pi

# Allow Flask to run in Jupyter
nest_asyncio.apply()

# Set current folder as base for templates and static
BASE_DIR = os.getcwd()
app = Flask(__name__, static_folder=os.path.join(BASE_DIR, "static"),
            template_folder=os.path.join(BASE_DIR, "templates"))

socketio = SocketIO(app, cors_allowed_origins="*", async_mode='threading')

# Home page
@app.route('/')
def index():
    return render_template('index.html')


# Endpoint to send command to robot
@app.route('/send', methods=['POST'])
def send():
    data = request.json
    command = data.get('command')
    socketio.emit('status', {'state': 'thinking', 'message': ' Dora is thinking...'})
    threading.Thread(target=send_and_update, args=(command,)).start()
    return jsonify({'ok': True})

# Function to handle sending command and updating GUI
def send_and_update(command):
    try:
        socketio.emit('status', {'state': 'thinking', 'message': ' Dora is thinking...'})
        response = send_command_to_pi(command)
        state = response.get('status', 'sad')
        message = response.get('message', '')

        # Friendly child-like messages
        if 'Comm error' in message:
            socketio.emit('status', {'state': 'sad', 'message': "Oops! I couldn’t find that item!"})
        else:
            socketio.emit('status', {'state': state, 'message': message})
    except Exception as e:
        socketio.emit('status', {'state': 'sad', 'message': "Uh-oh! Something went wrong."})

socketio.run(app, host='0.0.0.0', port=5000, debug=True, use_reloader=False, allow_unsafe_werkzeug=True)

Werkzeug appears to be used in a production deployment. Consider switching to a production web server instead.


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.231.63.90:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Dec/2025 14:08:59] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/lion.png HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/dora_idle.png HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/elephant.jpg HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /socket.io/?EIO=4&transport=polling&t=PhUYN5D HTTP/1.1" 200 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/tiger.png HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/zebra.jpg HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/giraffe.png HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/forest_bg.jpg HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "GET /static/leopard.jpg HTTP/1.1" 304 -
127.0.0.1 - - [02/Dec/2025 14:09:00] "POST /socket.io/?EIO=4&transport=polling&t